Setting up environmenet

In [1]:
%%capture
import os
import pickle
!pip install bertopic

In [2]:
# Data files (reviews, embeddings, BERTopic model) are read from the data/ folder
os.chdir(os.path.join(os.getcwd(), 'data')) if os.path.isdir('data') else None


loading sentence data

In [3]:
  # Define a filename for your pickled object
filename = 'df_cleaned_sentences.pkl'

# Load the object from the file
with open(filename, 'rb') as file:
    df = pickle.load(file)
print(f"DataFrame loaded from '{filename}' successfully!")

# Define a filename for your pickled object
filename = 'embeddings_sentence.pkl'

# Load the object from the file
with open(filename, 'rb') as file:
    embeddings_sentence = pickle.load(file)
print(f"Loaded  '{filename}' successfully!")

# Define a filename for your pickled object
filename = 'reduced_sentence_embeddings.pkl'

# Load the object from the file
with open(filename, 'rb') as file:
    reduced_embeddings_all = pickle.load(file)
print(f"Loaded  '{filename}' successfully!")

df.head()

DataFrame loaded from 'df_cleaned_sentences.pkl' successfully!
Loaded  'embeddings_sentence.pkl' successfully!
Loaded  'reduced_sentence_embeddings.pkl' successfully!


,User_ID,Original_Description,Sentence,Description,Cleaned_Description,sentiment_score
0,id10332,My husband and I have stayed in this hotel a f...,My husband and I have stayed in this hotel a f...,My husband and I have stayed in this hotel a f...,stayed hotel times,0.128171
1,id10332,My husband and I have stayed in this hotel a f...,"Though not the fanciest hotel, we love the fac...","Though not the fanciest hotel, we love the fac...",though hotel walk,0.947809
2,id10332,My husband and I have stayed in this hotel a f...,It is clean and the staff is very accomodating.,It is clean and the staff is very accomodating.,clean staff,0.946947
3,id10332,My husband and I have stayed in this hotel a f...,My only complaint is that the fan in the bathr...,My only complaint is that the fan in the bathr...,bathroom went much,-0.697226
4,id10332,My husband and I have stayed in this hotel a f...,We've stayed in pricier hotels which charged f...,We've stayed in pricier hotels which charged f...,stayed hotels internet breakfast,0.072468


load topic model

In [4]:

from bertopic import BERTopic


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [5]:

# Load the object from the file
topic_model = BERTopic.load("topic_model_sentence_uncleaned")

2026-03-24 10:56:02,657 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


cleaning topic structure

In [6]:
print(topic_model)

BERTopic(calculate_probabilities=False, ctfidf_model=ClassTfidfTransformer(...), embedding_model=<bertopic.backend._base.BaseEmbedder object at 0x7d32dd1fddc0>, hdbscan_model=<bertopic.cluster._base.BaseCluster object at 0x7d32daea4290>, language=None, low_memory=False, min_topic_size=10, n_gram_range=(1, 1), nr_topics=None, representation_model=None, seed_topic_list=None, top_n_words=50, umap_model=<bertopic.dimensionality._base.BaseDimensionalityReduction object at 0x7d33c174e8d0>, vectorizer_model=CountVectorizer(...), verbose=True, zeroshot_min_similarity=0.7, zeroshot_topic_list=None)


In [10]:
import json
topics_json_path = os.path.join(os.getcwd(), "topic_model_sentence_uncleaned", "topics.json")
topics_json = None
if os.path.exists(topics_json_path):
    with open(topics_json_path, "r", encoding="utf-8") as f:
        topics_json = json.load(f)


<_io.TextIOWrapper name='topic_model_sentence_uncleaned/topics.json' mode='r' encoding='utf-8'>


In [11]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4287,-1_hotel_hotels_rooms_hotel staff,"[hotel, hotels, rooms, hotel staff, staying, m...",NaN
1,0,829,0_good location_great location_near_restaurants,"[good location, great location, near, restaura...",NaN
2,1,796,1_little_room_room service_area,"[little, room, room service, area, downstairs,...",NaN
3,2,680,2_floor_bathroom_space_room,"[floor, bathroom, space, room, rooms, extremel...",NaN
4,3,468,3_room hotel_room_hotel room_hotel really,"[room hotel, room, hotel room, hotel really, s...",NaN
5,4,448,4_room wasn_different room_room_rooms,"[room wasn, different room, room, rooms, desk ...",NaN
6,5,323,5_bathroom small_room_rooms_bathroom,"[bathroom small, room, rooms, bathroom, bathro...",NaN
7,6,322,6_room_accomodating_housekeeping_day,"[room, accomodating, housekeeping, day, entire...",NaN
8,7,288,7_great location_liked hotel_hotels_recommend ...,"[great location, liked hotel, hotels, recommen...",NaN
9,8,277,8_rooms_hotel_room_nearby,"[rooms, hotel, room, nearby, outside, floor, o...",NaN


In [13]:
# Apply the trained BERTopic model to the new observations
topics_all, probabilities_all = topic_model.transform(df['Description'], embeddings_sentence)





2026-03-24 11:10:22,371 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


In [14]:
df['topic'] = topics_all
df['topic_probabilities'] = probabilities_all


In [15]:
df.head(10)


,User_ID,Original_Description,Sentence,Description,Cleaned_Description,sentiment_score,topic,topic_probabilities
0,id10332,My husband and I have stayed in this hotel a f...,My husband and I have stayed in this hotel a f...,My husband and I have stayed in this hotel a f...,stayed hotel times,0.128171,9,0.946478
1,id10332,My husband and I have stayed in this hotel a f...,"Though not the fanciest hotel, we love the fac...","Though not the fanciest hotel, we love the fac...",though hotel walk,0.947809,0,0.907792
2,id10332,My husband and I have stayed in this hotel a f...,It is clean and the staff is very accomodating.,It is clean and the staff is very accomodating.,clean staff,0.946947,6,0.929655
3,id10332,My husband and I have stayed in this hotel a f...,My only complaint is that the fan in the bathr...,My only complaint is that the fan in the bathr...,bathroom went much,-0.697226,8,0.904582
4,id10332,My husband and I have stayed in this hotel a f...,We've stayed in pricier hotels which charged f...,We've stayed in pricier hotels which charged f...,stayed hotels internet breakfast,0.072468,14,0.910561
5,id10332,My husband and I have stayed in this hotel a f...,Will stay there again.,Will stay there again.,stay,0.203629,12,0.925606
6,id10327,I stayed at the Crown Plaza April -- - April -...,I stayed at the Crown Plaza April -- - April -...,I stayed at the Crown Plaza April -- - April -...,stayed,0.080488,9,0.920264
7,id10327,I stayed at the Crown Plaza April -- - April -...,The staff was friendly and attentive.,The staff was friendly and attentive.,staff friendly,0.912853,6,0.974720
8,id10327,I stayed at the Crown Plaza April -- - April -...,The elevators are tiny (about -' by -').,The elevators are tiny (about -' by -').,,-0.050937,-1,0.890361
9,id10327,I stayed at the Crown Plaza April -- - April -...,The food in the restaurant was delicious but p...,The food in the restaurant was delicious but p...,food restaurant little,0.958273,1,0.920962


In [16]:
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))
topic_model.update_topics(df['Description'][:10000], n_gram_range=(1, 2),   vectorizer_model = vectorizer_model, top_n_words=30))

In [18]:
df.shape

(389758, 8)

In [19]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4287,-1_room_hotel_stay_great,"[room, hotel, stay, great, staff, rooms, clean...",NaN
1,0,829,0_location_walk_square_walking,"[location, walk, square, walking, distance, wa...",NaN
2,1,796,1_breakfast_restaurant_food_bar,"[breakfast, restaurant, food, bar, good, free,...",NaN
3,2,680,2_room_comfortable_bed_rooms,"[room, comfortable, bed, rooms, clean, beds, s...",NaN
4,3,468,3_hotel_stay_hotels_worst,"[hotel, stay, hotels, worst, hilton, service, ...",NaN
5,4,448,4_desk_room_check_told,"[desk, room, check, told, manager, called, rea...",NaN
6,5,323,5_shower_bathroom_tub_water,"[shower, bathroom, tub, water, toilet, bath, t...",NaN
7,6,322,6_staff_friendly_helpful_staff friendly,"[staff, friendly, helpful, staff friendly, fri...",NaN
8,7,288,7_hotel_great_great hotel_nice,"[hotel, great, great hotel, nice, hotel great,...",NaN
9,8,277,8_noise_hear_night_noisy,"[noise, hear, night, noisy, room, quiet, loud,...",NaN


Sentiment analysis per topic

In [32]:
sentiment_per_topic = df.groupby('topic')['sentiment_score'].mean().reset_index()
#sentiment_per_topic = sentiment_per_topic.sort_values(by='sentiment_score', ascending=False)
topic_names = topic_model.get_topic_info()[['Topic', 'Name']]
sentiment_per_topic_named = sentiment_per_topic.merge(topic_names, left_on='topic', right_on='Topic', how='left')
sentiment_per_topic_named = sentiment_per_topic_named.drop(columns=['Topic'])
sentiment_per_topic_named = sentiment_per_topic_named.rename(columns={'Name': 'Topic_Name'})
display(sentiment_per_topic_named)

,topic,sentiment_score,Topic_Name
0,-1,0.265121,-1_room_hotel_stay_great
1,0,0.515537,0_location_walk_square_walking
2,1,0.483893,1_breakfast_restaurant_food_bar
3,2,0.520034,2_room_comfortable_bed_rooms
4,3,-0.414088,3_hotel_stay_hotels_worst
5,4,-0.348475,4_desk_room_check_told
6,5,-0.130821,5_shower_bathroom_tub_water
7,6,0.857125,6_staff_friendly_helpful_staff friendly
8,7,0.746147,7_hotel_great_great hotel_nice
9,8,-0.213829,8_noise_hear_night_noisy


In [33]:
topic_model.get_topic(11)

[('experience', np.float64(0.054055387694849154)),
 ('happen', np.float64(0.045324361982373045)),
 ('don', np.float64(0.03822603820638735)),
 ('horrible', np.float64(0.03581760071510441)),
 ('just', np.float64(0.033575167116375204)),
 ('disgusting', np.float64(0.025899635418498882)),
 ('worth', np.float64(0.02442787781157754)),
 ('did', np.float64(0.023937764348818908)),
 ('ones', np.float64(0.023878400476736276)),
 ('don know', np.float64(0.02368546057003608)),
 ('things', np.float64(0.023600272600635172)),
 ('know', np.float64(0.02320183957940644)),
 ('little', np.float64(0.02222457490305804)),
 ('really', np.float64(0.021221603369560384)),
 ('disappointed', np.float64(0.02085486481035461)),
 ('weird', np.float64(0.020278993404441753)),
 ('unpleasant', np.float64(0.019966091675520374)),
 ('read reviews', np.float64(0.019683071971844466)),
 ('disappointing', np.float64(0.01918710064438931)),
 ('believe', np.float64(0.018390928078290767)),
 ('reviews', np.float64(0.018082181601983497))

predicting ratings from overall and topic level sentiment

In [34]:
import os
import pickle
import warnings
import json
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bertopic import BERTopic
from sklearn.decomposition import PCA
import statsmodels.api as sm

warnings.filterwarnings("ignore")


BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_pickle(path):
    with open(path, "rb") as file:
        obj = pickle.load(file)
    print(f"Loaded '{path}'")
    return obj


def assign_brand(review_text):
    text = str(review_text).lower()
    brand_terms = {
        # Hotel brands
        "brand_hilton": ["hilton"],
        "brand_marriott": ["marriott", "jw marriott"],
        "brand_sheraton": ["sheraton"],
        "brand_hyatt": ["hyatt", "park hyatt", "grand hyatt"],
        "brand_hi": [ "holiday inn"],
        "brand_novotel": ["novotel"],
    }

    for brand, terms in brand_terms.items():
        if any(term in text for term in terms):
            return brand
    return "unknown"


def build_topic_label(topic_model, topic_id, topics_json=None, n_words=3):
    words = []

    try:
        topic_terms = topic_model.get_topic(int(topic_id))
        if topic_terms:
            words = [term for term, _ in topic_terms[:n_words]]
    except Exception:
        pass

    if not words and topics_json is not None:
        rep = topics_json.get("topic_representations", {})
        topic_terms = rep.get(str(int(topic_id)), [])
        if topic_terms:
            words = [term_score[0] for term_score in topic_terms[:n_words]]

    if not words:
        return f"{int(topic_id)}"
    return f"{int(topic_id)}: " + " | ".join(words)


def wrap_topic_label(label, width=16):
    # Keep plot labels readable by inserting line breaks.
    return "\n".join(textwrap.wrap(label, width=width))


def save_heatmap(matrix, title, filename):
    n_topics = max(1, matrix.shape[1])
    heatmap_width = min(42, max(14, n_topics * 1.2))
    plt.figure(figsize=(heatmap_width, 7.5))
    sns.heatmap(matrix, cmap="RdBu_r", center=0)
    plt.title(title)
    plt.xlabel("Topic")
    plt.ylabel("Brand")
    plt.xticks(rotation=60, ha="right", fontsize=12)
    plt.yticks(fontsize=12, rotation=0, ha='right')
    plt.subplots_adjust(bottom=0.34)
    plt.tight_layout()
    plt.savefig(filename, dpi=180)
    plt.close()


def save_spider_plot(spider_df, topic_ids, topic_label_wrapped_map, title, filename):
    if spider_df.shape[0] < 2 or spider_df.shape[1] < 3:
        return

    angles = np.linspace(0, 2 * np.pi, spider_df.shape[1], endpoint=False).tolist()
    angles += angles[:1]

    data_min = float(spider_df.min().min())
    data_max = float(spider_df.max().max())
    radial_offset = max(0.0, -data_min)
    radial_upper = data_max + radial_offset
    if radial_upper <= 0:
        radial_upper = 0.1

    tick_min = np.floor(data_min * 10) / 10
    tick_max = np.ceil(data_max * 10) / 10
    tick_values = np.round(np.linspace(tick_min, tick_max, 5), 2)
    zero_radius = radial_offset

    fig = plt.figure(figsize=(12, 12))
    ax = plt.subplot(111, polar=True)
    for brand in spider_df.index:
        values = spider_df.loc[brand].tolist()
        radii = [value + radial_offset for value in values]
        radii += radii[:1]
        ax.plot(angles, radii, linewidth=2, label=brand)
        #ax.fill(angles, radii, alpha=0.1)

    zero_line = [zero_radius] * len(angles)
    ax.plot(angles, zero_line, color="black", linewidth=1.2, linestyle="--", label="zero")

    ax.set_ylim(0, radial_upper)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [topic_label_wrapped_map.get(int(t), f"{int(t)}") for t in topic_ids],
        fontsize=15,
    )
    ax.set_yticks(tick_values + radial_offset)
    ax.set_yticklabels([f"{tick:.2f}" for tick in tick_values], fontsize=15)
    ax.set_rlabel_position(90)
    #ax.set_title(title)
    fig.legend(
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        fontsize=15  # near top-right corner of figure
    )#    plt.tight_layout()
    ax.tick_params(axis="x", pad=45)
    #fig.subplots_adjust(top=1, bottom=0.1, left=0.1, right=1)
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.close()



def print_ols_outcomes(title, X, y, topic_model=None, topics_json=None):
    X_const = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X_const).fit()

    coef_df = pd.DataFrame({
        "feature": model.params.index,
        "coef": model.params.values,
        "se": model.bse.values,
        "t": model.tvalues.values,
        "p": model.pvalues.values,
    })

    # Sort by topic ID if features are topic indicators
    if all(f.startswith("pres_topic_") for f in coef_df["feature"] if f != "const"):
        coef_df["topic_id"] = coef_df["feature"].apply(
            lambda x: int(x.replace("pres_topic_", "")) if x.startswith("pres_topic_") else float("inf")
        )
        coef_df = coef_df.sort_values("topic_id").drop("topic_id", axis=1)

        # Build topic labels
        if topic_model is not None or topics_json is not None:
            topic_label_map = {
                int(f.replace("pres_topic_", "")): build_topic_label(topic_model, int(f.replace("pres_topic_", "")), topics_json=topics_json)
                for f in coef_df["feature"] if f.startswith("pres_topic_")
            }
            coef_df["feature"] = coef_df["feature"].apply(
                lambda x: topic_label_map.get(int(x.replace("pres_topic_", "")), x) if x.startswith("pres_topic_") else x
            )
    else:
        coef_df = coef_df.sort_values("feature")

    print(f"\n{title}")
    print(f"R^2: {model.rsquared:.6f}")
    print("Coefficients (with se, t and p):")
    print(coef_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
    return model





In [23]:
embeddings_sentence = load_pickle(os.path.join(BASE_DIR, "embeddings_sentence.pkl"))

df["brand"] = df["Description"].apply(assign_brand)

topic_dummies = pd.get_dummies(df["topic"], prefix="topic")

# Sentence-level topic sentiment vectors: zero everywhere, sentiment on assigned topic.
topic_sent_vectors = topic_dummies.mul(df["sentiment_score"].to_numpy().reshape(-1, 1))
topic_sent_vectors.columns = [c.replace("topic_", "sent_topic_") for c in topic_sent_vectors.columns]

# Sentence-level topic presence vectors: one on assigned topic, zero elsewhere.
topic_presence_vectors = topic_dummies.copy()
topic_presence_vectors.columns = [c.replace("topic_", "pres_topic_") for c in topic_presence_vectors.columns]



Loaded 'embeddings_sentence.pkl'


In [24]:
feature_frame = pd.concat(
    [
        df[["User_ID", "brand", "sentiment_score"]].reset_index(drop=True),
        topic_sent_vectors.reset_index(drop=True),
        topic_presence_vectors.reset_index(drop=True),
    ],
    axis=1,
)

review_level = (
    feature_frame.groupby(["User_ID"], as_index=False)
    .mean(numeric_only=True)
)
review_level = review_level.rename(columns={"sentiment_score": "review_sentiment"})


In [25]:
len(review_level)

38932

In [26]:
display(df.head())

,User_ID,Original_Description,Sentence,Description,Cleaned_Description,sentiment_score,topic,topic_probabilities,brand
0,id10332,My husband and I have stayed in this hotel a f...,My husband and I have stayed in this hotel a f...,My husband and I have stayed in this hotel a f...,stayed hotel times,0.128171,9,0.946478,unknown
1,id10332,My husband and I have stayed in this hotel a f...,"Though not the fanciest hotel, we love the fac...","Though not the fanciest hotel, we love the fac...",though hotel walk,0.947809,0,0.907792,unknown
2,id10332,My husband and I have stayed in this hotel a f...,It is clean and the staff is very accomodating.,It is clean and the staff is very accomodating.,clean staff,0.946947,6,0.929655,unknown
3,id10332,My husband and I have stayed in this hotel a f...,My only complaint is that the fan in the bathr...,My only complaint is that the fan in the bathr...,bathroom went much,-0.697226,8,0.904582,unknown
4,id10332,My husband and I have stayed in this hotel a f...,We've stayed in pricier hotels which charged f...,We've stayed in pricier hotels which charged f...,stayed hotels internet breakfast,0.072468,14,0.910561,unknown


In [27]:
csv_file_path = 'hotel-reviews.csv'

try:
  df_review = pd.read_csv(csv_file_path)
  print("CSV file imported successfully!")
  df_review.loc[[0, 6]] = df_review.loc[[6, 0]].values # change first review
except FileNotFoundError:
  print(f"Error: The file '{csv_file_path}' was not found. Please check the path.")
except Exception as e:
  print(f"An error occurred: {e}")


print(df_review.head())

CSV file imported successfully!
   User_ID                                        Description  \
0  id10332  My husband and I have stayed in this hotel a f...   
1  id10327  I stayed at the Crown Plaza April -- - April -...   
2  id10328  I booked this hotel through Hotwire at the low...   
3  id10329  Stayed here with husband and sons on the way t...   
4  id10330  My girlfriends and I stayed here to celebrate ...   

        Browser_Used Device_Used Is_Response  
0            Firefox      Tablet   not happy  
1  Internet Explorer      Mobile   not happy  
2            Mozilla      Tablet   not happy  
3   InternetExplorer     Desktop       happy  
4               Edge      Tablet   not happy  


In [28]:
df_review = df_review.merge(review_level, on='User_ID', how='left')


In [29]:
df_review.head()

,User_ID,Description,Browser_Used,Device_Used,Is_Response,review_sentiment,sent_topic_-1,sent_topic_0,sent_topic_1,sent_topic_2,...,pres_topic_6,pres_topic_7,pres_topic_8,pres_topic_9,pres_topic_10,pres_topic_11,pres_topic_12,pres_topic_13,pres_topic_14,pres_topic_15
0,id10332,My husband and I have stayed in this hotel a f...,Firefox,Tablet,not happy,0.266966,0.000000,0.157968,0.000000,0.000000,...,0.166667,0.000000,0.166667,0.166667,0.000000,0.000000,0.166667,0.000000,0.166667,0.0
1,id10327,I stayed at the Crown Plaza April -- - April -...,Internet Explorer,Mobile,not happy,-0.136469,-0.002830,-0.009301,0.053237,0.033152,...,0.055556,0.000000,0.055556,0.055556,0.055556,0.166667,0.055556,0.000000,0.055556,0.0
2,id10328,I booked this hotel through Hotwire at the low...,Mozilla,Tablet,not happy,-0.264112,-0.035025,0.001764,0.009258,0.018734,...,0.000000,0.052632,0.000000,0.000000,0.000000,0.105263,0.000000,0.052632,0.000000,0.0
3,id10329,Stayed here with husband and sons on the way t...,InternetExplorer,Desktop,happy,0.787707,0.121781,0.169242,0.240698,0.117368,...,0.000000,0.125000,0.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
4,id10330,My girlfriends and I stayed here to celebrate ...,Edge,Tablet,not happy,0.107817,0.000000,0.000000,0.000000,0.035208,...,0.055556,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.055556,0.0


In [36]:
all_topics = sorted(df['topic'].unique())

topic_label_map = {
    int(t): build_topic_label(topic_model, int(t), topics_json=topics_json)
    for t in all_topics
}
topic_label_wrapped_map = {
    topic_id: wrap_topic_label(label)
    for topic_id, label in topic_label_map.items()
}

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import statsmodels.api as sm

pres_cols = [c for c in df_review.columns if c.startswith("pres_topic_")]
pres_cols = pres_cols[1:] # Exclude pres_topic_-1 for now as it's often the 'other' category
sent_cols = [c for c in df_review.columns if c.startswith("sent_topic_")]

# Combine presence and sentiment columns for the logit model
x_topic = df_review[pres_cols + sent_cols]

# Create custom labels for the columns
custom_labels = {}
for col in pres_cols:
    topic_id = int(col.replace("pres_topic_", ""))
    custom_labels[col] = f"Presence: {topic_label_wrapped_map.get(topic_id, str(topic_id))}"
for col in sent_cols:
    topic_id = int(col.replace("sent_topic_", ""))
    custom_labels[col] = f"Sentiment: {topic_label_wrapped_map.get(topic_id, str(topic_id))}"

x_topic_labeled = x_topic.rename(columns=custom_labels)

x_train = x_topic_labeled[:5000]
y_train = (df_review[:5000]['Is_Response']=="happy")

# Add a constant to the features matrix for statsmodels
x_train = sm.add_constant(x_train)

# Fit the Logit model
logit_model = sm.Logit(y_train, x_train)
logit_result = logit_model.fit()

csv_summary = logit_result.summary().as_csv()
summary_path = os.path.join(OUTPUT_DIR, "logit_model_summary_both.csv")
with open(summary_path, "w") as f:
    f.write(csv_summary)
print(f"Logit model summary exported to {summary_path}")


Optimization terminated successfully.
         Current function value: 0.274844
         Iterations 8
Logit model summary exported to outputs/logit_model_summary_both.csv


In [38]:
logit_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            Is_Response   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4966
Method:                           MLE   Df Model:                           33
Date:                Tue, 24 Mar 2026   Pseudo R-squ.:                  0.5607
Time:                        11:37:52   Log-Likelihood:                -1374.2
converged:                       True   LL-Null:                       -3128.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================================================
                                                         coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
const                                                 -0.4991      0.486     -1.028      0.304      -1.451       0.453
Presence: 0: location |
walk | square                  1.5361      0.709      2.167      0.030       0.147       2.925
Presence: 1: breakfast |
restaurant |
food             0.5237      0.701      0.747      0.455      -0.850       1.897
Presence: 2: room |
comfortable |
bed                 -1.5997      0.738     -2.168      0.030      -3.046      -0.154
Presence: 3: hotel | stay
| hotels                    -3.3016      0.689     -4.789      0.000      -4.653      -1.950
Presence: 4: desk | room |
check                      -0.0314      0.814     -0.039      0.969      -1.627       1.564
Presence: 5: shower |
bathroom | tub                  -1.0293      0.885     -1.164      0.245      -2.763       0.704
Presence: 6: staff |
friendly |
helpful                0.5965      1.937      0.308      0.758      -3.200       4.393
Presence: 7: hotel | great
| great hotel               0.5273      1.021      0.516      0.606      -1.474       2.528
Presence: 8: noise | hear
| night                     -0.1132      0.876     -0.129      0.897      -1.830       1.604
Presence: 9: stayed |
nights | stayed
nights           0.4679      0.925      0.506      0.613      -1.345       2.281
Presence: 10: pool |
fitness | gym                    -2.1198      1.100     -1.928      0.054      -4.275       0.035
Presence: 11: experience |
happen | don               -1.0010      1.033     -0.969      0.333      -3.026       1.024
Presence: 12: stay |
definitely |
definitely stay     -0.3148      1.152     -0.273      0.785      -2.572       1.942
Presence: 13: parking |
valet | car                    1.4484      0.929      1.559      0.119      -0.373       3.269
Presence: 14: internet |
free | wifi                   0.3106      0.979      0.317      0.751      -1.608       2.229
Presence: 15: highly |
nice | highly
recommended       1.8564      1.208      1.536      0.124      -0.512       4.224
Sentiment: -1: room | hotel
| stay                     5.7909      0.894      6.477      0.000       4.038       7.543
Sentiment: 0: location |
walk | square                 1.7921      0.747      2.399      0.016       0.328       3.256
Sentiment: 1: breakfast |
restaurant |
food            3.9560      0.773      5.115      0.000       2.440       5.472
Sentiment: 2: room |
comfortable |
bed                 7.8376      0.743     10.542      0.000       6.380       9.295
Sentiment: 3: hotel | stay
| hotels                    3.9793      0.798      4.984      0.000       2.414       5.544
Sentiment: 4: desk | room |
check                      6.9712      1.298      5.370      0.000       4.427       9.516
Sentiment: 5: shower |
bathroom | tub                  9.1605      1.282      7.145      0.000       6.648      11.673
Sentiment: 6: staff |
friend

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import statsmodels.api as sm

pres_cols = [c for c in df_review.columns if c.startswith("pres_topic_")]
pres_cols = pres_cols[1:] # Exclude pres_topic_-1 for now as it's often the 'other' category, to avoid multicollinearity
sent_cols = [c for c in df_review.columns if c.startswith("sent_topic_")]

# focus on relevant attributes:
pres_cols = [ pres_cols[0],  pres_cols[1], pres_cols[2], pres_cols[4], pres_cols[5], pres_cols[6], pres_cols[8], pres_cols[10], pres_cols[13], pres_cols[14]]
sent_cols = sent_cols[1:] # Exclude pres_topic_-1 also from sentiments first
sent_cols = [ sent_cols[0],  sent_cols[1], sent_cols[2], sent_cols[4], sent_cols[5], sent_cols[6], sent_cols[8], sent_cols[10], sent_cols[13], sent_cols[14]]

# Combine presence and sentiment columns for the logit model
x_topic = df_review[pres_cols + sent_cols]

# Create custom labels for the columns
custom_labels = {}
for col in pres_cols:
    topic_id = int(col.replace("pres_topic_", ""))
    custom_labels[col] = f"Presence: {topic_label_wrapped_map.get(topic_id, str(topic_id))}"
for col in sent_cols:
    topic_id = int(col.replace("sent_topic_", ""))
    custom_labels[col] = f"Sentiment: {topic_label_wrapped_map.get(topic_id, str(topic_id))}"

x_topic_labeled = x_topic.rename(columns=custom_labels)

x_train = x_topic_labeled[:5000]
y_train = (df_review[:5000]['Is_Response']=="happy")

# Add a constant to the features matrix for statsmodels
x_train = sm.add_constant(x_train)

# Fit the Logit model
logit_model = sm.Logit(y_train, x_train)
logit_result = logit_model.fit()

csv_summary = logit_result.summary().as_csv()
summary_path = os.path.join(OUTPUT_DIR, "logit_model_summary_both_subset.csv")
with open(summary_path, "w") as f:
    f.write(csv_summary)
print(f"Logit model summary exported to {summary_path}")


Optimization terminated successfully.
         Current function value: 0.369601
         Iterations 7
Logit model summary exported to outputs/logit_model_summary_both_subset.csv


TypeError: list indices must be integers or slices, not tuple

In [46]:
logit_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            Is_Response   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4979
Method:                           MLE   Df Model:                           20
Date:                Tue, 24 Mar 2026   Pseudo R-squ.:                  0.4093
Time:                        12:02:31   Log-Likelihood:                -1848.0
converged:                       True   LL-Null:                       -3128.3
Covariance Type:            nonrobust   LLR p-value:                     0.000
===============================================================================================================
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
const                                           0.6644      0.101      6.572      0.000       0.466       0.863
Presence: 0: location |
walk | square           0.4585      0.455      1.008      0.314      -0.433       1.350
Presence: 1: breakfast |
restaurant |
food     -0.8737      0.482     -1.813      0.070      -1.818       0.071
Presence: 2: room |
comfortable |
bed          -3.3586      0.493     -6.806      0.000      -4.326      -2.391
Presence: 4: desk | room |
check               -2.4163      0.571     -4.229      0.000      -3.536      -1.296
Presence: 5: shower |
bathroom | tub           -3.3803      0.669     -5.055      0.000      -4.691      -2.070
Presence: 6: staff |
friendly |
helpful         0.0069      1.774      0.004      0.997      -3.470       3.483
Presence: 8: noise | hear
| night              -2.5862      0.625     -4.137      0.000      -3.812      -1.361
Presence: 10: pool |
fitness | gym             -3.2735      0.960     -3.411      0.001      -5.154      -1.393
Presence: 13: parking |
valet | car             0.2811      0.685      0.411      0.681      -1.061       1.623
Presence: 14: internet |
free | wifi           -1.5659      0.799     -1.960      0.050      -3.131      -0.000
Sentiment: 0: location |
walk | square          3.1279      0.678      4.616      0.000       1.800       4.456
Sentiment: 1: breakfast |
restaurant |
food     6.6799      0.728      9.170      0.000       5.252       8.108
Sentiment: 2: room |
comfortable |
bed         10.4313      0.698     14.945      0.000       9.063      11.799
Sentiment: 4: desk | room |
check              10.0028      1.168      8.567      0.000       7.714      12.291
Sentiment: 5: shower |
bathroom | tub          11.9015      1.164     10.224      0.000       9.620      14.183
Sentiment: 6: staff |
friendly |
helpful        8.1861      1.993      4.108      0.000       4.281      12.092
Sentiment: 8: noise | hear
| night              6.1288      1.253      4.892      0.000       3.674       8.584
Sentiment: 10: pool |
fitness | gym             9.2518      1.523      6.075      0.000       6.267      12.237
Sentiment: 13: parking |
valet | car            3.9049      1.349      2.894      0.004       1.261       6.549
Sentiment: 14: internet |
free | wifi           5.2160      1.395      3.740      0.000       2.483       7.949
===============================================================================================================
"""

In [ ]:
sent_cols = [c for c in df_review.columns if c.startswith("sent_topic_")]

# Combine presence and sentiment columns for the logit model
x_topic = df_review[sent_cols]

# Create custom labels for the columns
custom_labels = {}
for col in sent_cols:
    topic_id = int(col.replace("sent_topic_", ""))
    custom_labels[col] = f"Sentiment: {topic_label_wrapped_map.get(topic_id, str(topic_id))}"

x_topic_labeled = x_topic.rename(columns=custom_labels)

x_train = x_topic_labeled[:5000]
y_train = (df_review[:5000]['Is_Response']=="happy")

# Add a constant to the features matrix for statsmodels
x_train = sm.add_constant(x_train)

# Fit the Logit model
logit_model = sm.Logit(y_train, x_train)
logit_result = logit_model.fit()

csv_summary = logit_result.summary().as_csv()
summary_path = os.path.join(OUTPUT_DIR, "logit_model_summary_sentiments.csv")
with open(summary_path, "w") as f:
    f.write(csv_summary)
print(f"Logit model summary exported to {summary_path}")


In [ ]:
logit_result.summary()

In [ ]:
pres_cols = [c for c in df_review.columns if c.startswith("pres_topic_")]
pres_cols = pres_cols[1:] # Exclude pres_topic_-1 for now as it's often the 'other' category

# Combine presence and sentiment columns for the logit model
x_topic = df_review[pres_cols ]

# Create custom labels for the columns
custom_labels = {}
for col in pres_cols:
    topic_id = int(col.replace("pres_topic_", ""))
    custom_labels[col] = f"Presence: {topic_label_wrapped_map.get(topic_id, str(topic_id))}"

x_topic_labeled = x_topic.rename(columns=custom_labels)

x_train = x_topic_labeled[:5000]
y_train = (df_review[:5000]['Is_Response']=="happy")

# Add a constant to the features matrix for statsmodels
x_train = sm.add_constant(x_train)

# Fit the Logit model
logit_model = sm.Logit(y_train, x_train)
logit_result = logit_model.fit()

csv_summary = logit_result.summary().as_csv()
summary_path = os.path.join(OUTPUT_DIR, "logit_model_summary_presence.csv")
with open(summary_path, "w") as f:
    f.write(csv_summary)
print(f"Logit model summary exported to {summary_path}")


In [ ]:
logit_result.summary()

In [ ]:
x_topic_test = x_topic[5000:]
# Add a constant to the test features, similar to the training data
x_topic_test_with_constant = sm.add_constant(x_topic_test, has_constant='add')

# Predict probabilities using the fitted logit model
y_pred_proba = logit_result.predict(x_topic_test_with_constant)

# Convert probabilities to binary predictions (True/False)
y_pred = (y_pred_proba > 0.5).astype(bool)

y_test = (df_review[5000:]['Is_Response'] == "happy")

def evaluate_performance(y_true, y_pred):
    """Create and print the classification report"""
    performance = classification_report(
        y_true, y_pred,
        target_names=["Not Happy Review", "Happy Review"]
    )
    print(performance)

evaluate_performance(y_test, y_pred)

In [ ]:

# Topic sentiment by brand (sentence-level average sentiment inside each topic/brand cell).
brand_topic_sent = (
    df.groupby(["brand", "topic"], as_index=False)["sentiment_score"]
    .mean()
    .rename(columns={"sentiment_score": "avg_sentiment"})
)

pivot_brand_topic = brand_topic_sent.pivot(index="brand", columns="topic", values="avg_sentiment")
pivot_brand_topic = pivot_brand_topic.fillna(0.0)

topic_label_map = {
    int(t): build_topic_label(topic_model, int(t), topics_json=topics_json)
    for t in pivot_brand_topic.columns
}
topic_label_wrapped_map = {
    topic_id: wrap_topic_label(label)
    for topic_id, label in topic_label_map.items()
}
pivot_brand_topic_labeled = pivot_brand_topic.copy()
pivot_brand_topic_labeled.columns = [topic_label_wrapped_map[int(t)] for t in pivot_brand_topic.columns]

pivot_brand_topic_demeaned = pivot_brand_topic.sub(pivot_brand_topic.mean(axis=0), axis=1)
pivot_brand_topic_demeaned_labeled = pivot_brand_topic_demeaned.copy()
pivot_brand_topic_demeaned_labeled.columns = [
    topic_label_wrapped_map[int(t)] for t in pivot_brand_topic_demeaned.columns
]


save_heatmap(
    pivot_brand_topic_labeled,
    "Average Topic Sentiment by Brand",
    os.path.join(OUTPUT_DIR, "brand_topic_sentiment_heatmap.png"),
)
save_heatmap(
    pivot_brand_topic_demeaned_labeled,
    "Topic-Demeaned Sentiment by Brand",
    os.path.join(OUTPUT_DIR, "brand_topic_sentiment_demeaned_heatmap.png"),
)



In [ ]:
# Perceptual map of brands using PCA on the topic sentiment matrix.
if pivot_brand_topic.shape[0] >= 2 and pivot_brand_topic.shape[1] >= 2:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(pivot_brand_topic.values)
    pca_df = pd.DataFrame(coords, columns=["pc1", "pc2"], index=pivot_brand_topic.index)

    plt.figure(figsize=(8, 6))
    plt.axhline(0, color="grey", linewidth=0.6)
    plt.axvline(0, color="grey", linewidth=0.6)
    plt.scatter(pca_df["pc1"], pca_df["pc2"], s=90)
    for idx, row in pca_df.iterrows():
        plt.text(row["pc1"] + 0.01, row["pc2"] + 0.01, idx)
    plt.title("Brand Perceptual Map (PCA of Topic Sentiment Profiles)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "brand_perceptual_map.png"), dpi=180)
    plt.close()

# Spider chart for all topics and major brands.
brand_counts = df["brand"].value_counts()
top_brands = [b for b in brand_counts.index if b != "unknown"][:4]
all_topics = sorted([int(t) for t in pivot_brand_topic.columns])

spider_df = pivot_brand_topic.loc[[b for b in top_brands if b in pivot_brand_topic.index], all_topics]
save_spider_plot(
    spider_df,
    all_topics,
    topic_label_wrapped_map,
    "Spider Plot: Brand Topic Sentiment (All Topics)",
    os.path.join(OUTPUT_DIR, "brand_topic_sentiment_spider.png"),
)

spider_df_demeaned = pivot_brand_topic_demeaned.loc[
    [b for b in top_brands if b in pivot_brand_topic_demeaned.index], all_topics
]
save_spider_plot(
    spider_df_demeaned,
    all_topics,
    topic_label_wrapped_map,
    "Spider Plot: Topic-Demeaned Brand Sentiment (All Topics)",
    os.path.join(OUTPUT_DIR, "brand_topic_sentiment_demeaned_spider.png"),
)

print(f"\nDone. Outputs saved in: {OUTPUT_DIR}")

